# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AhmedElwhaiby/FlyRank-Inten-Repo"
REPO_DIR = "FlyRank-Inten-Repo"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 4 — CTR / Engagement Opportunity Scoring.**

Question: *Which visible pages under-capture clicks or engagement, and deserve metadata,
content, or monitoring review?*

I'm picking this lane because it has three things going for it: (1) the signal is dense —
GSC impressions/clicks are the best-populated metrics in the dataset, so I'm not fighting
sparsity like the AI-referral freestyle direction would; (2) a naive rule (flat "low CTR = bad")
is provably wrong here, since CTR depends on position — which makes the position-tier-adjusted
gap a genuinely non-trivial thing to get right, not a lookup table dressed up as ML; and
(3) the output is directly actionable — a ranked queue with reason codes a reviewer can act on
this week, not just an interesting pattern.

In [9]:
lane4_cols = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
              "sessions_90d", "engagement_rate", "scroll_rate", "main_intent"]

print("Rows:", len(df))
print("\nColumn presence + non-null counts for Lane 4's key fields:")
print(df[lane4_cols].notna().sum())

Rows: 30000

Column presence + non-null counts for Lane 4's key fields:
impressions_90d    30000
clicks_90d         30000
ctr                30000
avg_position       30000
sessions_90d       30000
engagement_rate    30000
scroll_rate        29875
main_intent        27626
dtype: int64


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision improved:** out of all visible pages, which N should a content reviewer look at
first, because they're capturing fewer clicks/less engagement than their position and intent
would predict?

**Who acts, and how:** a content/SEO reviewer with limited weekly capacity (roughly 20-50 pages).
For each flagged page they rewrite the title/meta, improve the snippet, fix an intent mismatch,
improve on-page engagement, or decide it's fine and move on.

**Cost of a wrong call:**
- False positive (flagged but fine) — wasted reviewer time. Cheap but not free; it burns
  limited review capacity.
- False negative (a real high-impression under-performer never surfaces) — a live opportunity
  stays broken. More expensive, especially at high volume.

This asymmetry is why precision@K (K = the reviewer's real weekly capacity) is the metric that
matters, not generic accuracy — the reviewer only ever looks at the top K anyway.

In [10]:
# How big is the pool this decision applies to?
eligible = df[(df["impressions_90d"] > 0) & (df["avg_position"] > 0)]

print("Total rows:", len(df))
print("Eligible (impressions_90d > 0 and avg_position > 0):", len(eligible))
print("Eligible with meaningful volume (impressions_90d >= 500):",
      (eligible["impressions_90d"] >= 500).sum())

Total rows: 30000
Eligible (impressions_90d > 0 and avg_position > 0): 28795
Eligible with meaningful volume (impressions_90d >= 500): 16726


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

The numbers below need to show two things: that avg_position really does need to be handled
carefully (0 means no data, not rank zero), and that CTR really does vary by position tier
(the whole justification for this lane over a flat rule).

In [11]:
# Number 1: avg_position=0 is common enough to matter -- must be filtered, not treated as rank 0
print("Rows with avg_position == 0 (no data, not rank 0):", (df["avg_position"] == 0).sum())

# Number 2: CTR clearly varies by position tier -- this is the whole case for this lane
valid = df[df["avg_position"] > 0].copy()
valid["position_tier"] = pd.cut(
    valid["avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)
print("\nMedian CTR by position tier:")
print(valid.groupby("position_tier", observed=True)["ctr"].median())

# Number 3: how many visible, high-impression pages sit below their tier's median CTR?
# (rough cut -- the real gap score comes later, this just sizes the opportunity)
tier_median = valid.groupby("position_tier", observed=True)["ctr"].transform("median")
below_tier_median = valid[(valid["impressions_90d"] >= 500) & (valid["ctr"] < tier_median)]
print("\nHigh-impression pages below their tier's median CTR:", len(below_tier_median))

Rows with avg_position == 0 (no data, not rank 0): 1205

Median CTR by position tier:
position_tier
1-3      0.00
4-10     0.16
11-20    0.10
21+      0.00
Name: ctr, dtype: float64

High-impression pages below their tier's median CTR: 3659


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- Observed: which pages have a CTR or engagement gap relative to others at the same position
  and intent, on this anonymized 30k-row slice (or the warehouse release, if I move to it).
- Decision-support: a ranked list, with reason codes, for a reviewer to check first — not a
  guarantee any of them are actually broken.
- Directional: broad patterns in which content types or intents show gaps more often.

**What I will never claim:**
- That a low-CTR page is *caused* by a bad title/meta — I have no experiment, only an
  association.
- That fixing a flagged page *will* increase its clicks — that requires a causal design
  (before/after test or holdout), which this lane does not run by default.
- Anything about Google's ranking algorithm, or AI citations/rankings.
- Any claim resting on `health_score`, `priority_score`, or other product decision flags as
  if they were ground truth — they aren't even in this dataset, on purpose.

In [12]:
frame = (
    "For the content reviewer with limited weekly review capacity, deciding which visible "
    "pages to review first for title/meta/engagement fixes, we will build a ranked list of "
    "pages scored by how far their observed CTR/engagement falls below the expected value "
    "for their position tier and intent, from the starter dataset, predicting/scoring a "
    "position-tier-adjusted CTR/engagement gap, measured by precision@K on the top K "
    "candidates, sanity-checked by hand. A wrong call costs wasted reviewer time (false "
    "positive) or a missed high-impression opportunity (false negative). A plain rule isn't "
    "enough because raw CTR thresholds ignore position, and position/intent/content-type may "
    "interact in ways a flat rule misses. We will claim only observed / decision-support "
    "results -- never that a fix caused a click increase, since that requires an experiment "
    "this data can't give us."
)
print(frame)

For the content reviewer with limited weekly review capacity, deciding which visible pages to review first for title/meta/engagement fixes, we will build a ranked list of pages scored by how far their observed CTR/engagement falls below the expected value for their position tier and intent, from the starter dataset, predicting/scoring a position-tier-adjusted CTR/engagement gap, measured by precision@K on the top K candidates, sanity-checked by hand. A wrong call costs wasted reviewer time (false positive) or a missed high-impression opportunity (false negative). A plain rule isn't enough because raw CTR thresholds ignore position, and position/intent/content-type may interact in ways a flat rule misses. We will claim only observed / decision-support results -- never that a fix caused a click increase, since that requires an experiment this data can't give us.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.